In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd
import numpy as np

In [45]:
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

train_df = pd.read_csv("./data/df_train.csv")
test_df = pd.read_csv("./data/df_test.csv")

train_df.drop(["month", "date"], axis=1, inplace=True)
test_df.drop(["month", "date"], axis=1, inplace=True)

train_df = train_df.assign(**train_df.select_dtypes(bool).astype(int))
test_df = test_df.assign(**test_df.select_dtypes(bool).astype(int))

scaler_train = MinMaxScaler()
scaler_test= MinMaxScaler()
train_np = scaler_train.fit_transform(train_df)
test_np = scaler_test.fit_transform(test_df)

train = pd.DataFrame(train_np, columns=train_df.columns)
test = pd.DataFrame(test_np, columns=test_df.columns)

X_train = torch.tensor(train.drop("price", axis=1).values, dtype=torch.float)
y_train = torch.tensor(train["price"].values, dtype=torch.float)

X_test = torch.tensor(test.drop("price", axis=1).values, dtype=torch.float)
y_test = torch.tensor(test["price"].values, dtype=torch.float)

m = X_train.shape[1]


In [50]:
class LinearReg(nn.Module):
    def __init__(self,):
        super().__init__()
        self.lin = nn.Linear( m, 1)
        
    def forward(self, x):
        y = self.lin(x)
        return y          

model = LinearReg()
criterion  = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)  
num_epochs = 200

for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    output = model(X_train)
    loss = criterion(output, y_train)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1)%50 ==0: 
        print(f"epoch: {epoch+1} --> loss: {loss.item()}")


    

c:\ProgramData\anaconda3\lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([13603])) that is different to the input size (torch.Size([13603, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch: 50 --> loss: 0.03817838802933693
epoch: 100 --> loss: 0.03735344484448433
epoch: 150 --> loss: 0.03727574273943901
epoch: 200 --> loss: 0.03726259991526604


In [51]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train.numpy(), y_train.numpy())

sk_weights = sk_model.coef_
sk_bias = sk_model.intercept_

my_weights = model.lin.weight.detach().numpy().flatten()
my_bias = model.lin.bias.item()

print(f"my bias: {my_bias} && sk_bias :{sk_bias}")
print(f"my weights:\n {my_weights} \n && \n sk_weights : \n{sk_weights}")





my bias: 0.3665086328983307 && sk_bias :0.04176443815231323
my weights:
 [-2.0595279e-03 -4.8821457e-03 -5.5169454e-04  1.0579453e-02
 -2.2513086e-04 -3.7766586e-04 -1.3449155e-04 -1.0054569e-03
 -6.4010630e-05  2.0573456e-04  3.4433464e-04] 
 && 
 sk_weights : 
[-0.00232012  0.18688092  0.02032876  0.3450942   0.05947022  0.09709001
  0.06135552 -0.02505395 -0.02490442 -0.01732697  0.28179416]


In [52]:
model.eval()

outputs = model(X_test) 


loss = criterion(outputs.flatten(), y_test)


rmse = torch.sqrt(loss).item()

print(f"Test Loss: {loss.item():.4f}, RMSE: {rmse:.4f}")

y_pred_sk = sk_model.predict(X_test.numpy())
mse_sk = mean_squared_error(y_test.numpy(), y_pred_sk)
rmse_sk = np.sqrt(mse_sk)

print(f"Sklearn Test MSE: {mse_sk:.4f}, RMSE: {rmse_sk:.4f}")

Test Loss: 0.0370, RMSE: 0.1925
Sklearn Test MSE: 0.0096, RMSE: 0.0981


In [ ]:
to